# 01 - Industrial Quality Inspection: Multi-Dataset Exploration
**Authors:** Tanya Tejaswi & Gayatri U

### Objectives:
1. Explore normal reference units and defective parts across the **8-Line Industrial Quality Suite**:
   - **pcb1**, **pcb2**, **pcb3**, **pcb4**, **capsules** (Amazon VisA)
   - **bottle**, **cable** (MVTec AD)
   - **steel** (NEU Surface Defect)
2. Compile structured metadata across 5,800+ raw images.
3. Inspect defect types, label balance, and spatial distributions.

In [ ]:
import os
import sys
import tempfile
os.environ["MPLCONFIGDIR"] = tempfile.gettempdir()
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.append("..")
from src.preprocessing import build_industrial_suite_metadata

print("✅ Loaded preprocessing tools")

### 1. Build & Index Master Industrial Metadata
We scan the dataset directories (`datasets/visa`, `data/mvtec`, `data/steel`) and generate structured metadata.

In [ ]:
# Build or load industrial suite metadata
meta_csv = Path("../data/embeddings/metadata_full.csv")
if meta_csv.exists():
    metadata_df = pd.read_csv(meta_csv)
else:
    metadata_df = build_industrial_suite_metadata("..")
    metadata_df.to_csv(meta_csv, index=False)

print(f"📊 Indexed {len(metadata_df)} images across {metadata_df['category'].nunique()} industrial categories.")
summary = pd.crosstab(metadata_df['category'], [metadata_df['split'], metadata_df['is_anomaly']])
print("\n=== Dataset Split & Defect Distribution ===")
print(summary)

### 2. Visualize Samples across all 8 Manufacturing Lines
Display inspection examples for each line.

In [ ]:
categories = ["pcb1", "pcb2", "pcb3", "pcb4", "capsules", "bottle", "cable", "steel"]
fig, axes = plt.subplots(len(categories), 2, figsize=(8, 2.5 * len(categories)), dpi=120)

for idx, cat in enumerate(categories):
    cat_df = metadata_df[metadata_df['category'] == cat]
    good_sample = cat_df[cat_df['is_anomaly'] == 0].iloc[0]
    defect_sample = cat_df[cat_df['is_anomaly'] == 1].iloc[0]
    
    img_good = Image.open(Path("..") / good_sample['image_path']).convert("RGB").resize((180, 180))
    img_defect = Image.open(Path("..") / defect_sample['image_path']).convert("RGB").resize((180, 180))
    
    axes[idx, 0].imshow(img_good)
    axes[idx, 0].set_title(f"{cat.upper()} - Normal", fontsize=9, fontweight='bold', color='#059669')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(img_defect)
    axes[idx, 1].set_title(f"{cat.upper()} - {defect_sample['defect_type'].upper()}", fontsize=9, fontweight='bold', color='#dc2626')
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()